# 02 — Model Training
Treina ChurnModel e ExpansionModel, salva pkl em `ml/models/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from pathlib import Path

from decision_service.models   import ChurnModel, ExpansionModel
from decision_service.dataset  import ChurnDataset
from decision_service.training import ChurnTrainer, ExpansionTrainer

MODEL_DIR = Path('../ml/models')
DATA_PATH = Path('../data/train_dataset.csv')

print(f'Dataset: {DATA_PATH}')
print(f'Models : {MODEL_DIR}')

## 1. Treinar ChurnModel

In [ ]:
churn_config = ChurnModel()
print('Features do ChurnModel:')
for f in churn_config.features:
    print(f'  - {f}')

In [ ]:
churn_ds = ChurnDataset(DATA_PATH, churn_config)
X_churn, y_churn, churn_scaler, churn_encoder = churn_ds.prepare()
churn_features = churn_ds.feature_names()

print(f'X shape: {X_churn.shape}')
print(f'y distribution: {pd.Series(y_churn).value_counts().to_dict()}')

In [ ]:
churn_trainer = ChurnTrainer(churn_config)
churn_model, churn_report = churn_trainer.train(X_churn, y_churn, churn_features)

print(f'\nAUC: {churn_report.auc}')
print(f'Avg Precision: {churn_report.avg_precision}')

In [ ]:
# Salvar modelo, scaler e encoder
MODEL_DIR.mkdir(parents=True, exist_ok=True)

churn_trainer.save(MODEL_DIR / 'churn_model.pkl')
ChurnDataset.save_scaler(churn_scaler,   MODEL_DIR / 'churn_scaler.pkl')
if churn_encoder:
    ChurnDataset.save_encoder(churn_encoder, MODEL_DIR / 'churn_encoder.pkl')

print('Churn model salvo!')

## 2. Treinar ExpansionModel

In [ ]:
exp_config = ExpansionModel()
exp_ds     = ChurnDataset(DATA_PATH, exp_config)
X_exp, y_exp, exp_scaler, exp_encoder = exp_ds.prepare()
exp_features = exp_ds.feature_names()

print(f'X shape: {X_exp.shape}')
print(f'y distribution (upsell): {pd.Series(y_exp).value_counts().to_dict()}')

In [ ]:
exp_trainer = ExpansionTrainer(exp_config)
exp_model, exp_report = exp_trainer.train(X_exp, y_exp, exp_features)

exp_trainer.save(MODEL_DIR / 'expansion_model.pkl')
ChurnDataset.save_scaler(exp_scaler, MODEL_DIR / 'expansion_scaler.pkl')

print('Expansion model salvo!')

## 3. Feature Importance — Churn

In [ ]:
import matplotlib.pyplot as plt

importances = churn_model.feature_importances_
feat_imp    = pd.DataFrame({'feature': churn_features, 'importance': importances})
feat_imp    = feat_imp.sort_values('importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(feat_imp['feature'], feat_imp['importance'], color='#e74c3c')
plt.xlabel('Importance')
plt.title('Feature Importance — ChurnModel')
plt.tight_layout()
plt.show()

print(feat_imp.sort_values('importance', ascending=False).to_string(index=False))

## 4. Feature Importance — Expansion

In [ ]:
importances_exp = exp_model.feature_importances_
feat_imp_exp    = pd.DataFrame({'feature': exp_features, 'importance': importances_exp})
feat_imp_exp    = feat_imp_exp.sort_values('importance', ascending=True)

plt.figure(figsize=(9, 5))
plt.barh(feat_imp_exp['feature'], feat_imp_exp['importance'], color='#3498db')
plt.xlabel('Importance')
plt.title('Feature Importance — ExpansionModel')
plt.tight_layout()
plt.show()

## 5. Resumo

In [ ]:
print('='*50)
print('  RESUMO DO TREINAMENTO')
print('='*50)
print(f'  ChurnModel     AUC: {churn_report.auc:.4f}  |  AvgPrec: {churn_report.avg_precision:.4f}')
print(f'  ExpansionModel AUC: {exp_report.auc:.4f}  |  AvgPrec: {exp_report.avg_precision:.4f}')
print('='*50)
print(f'  Arquivos salvos em: {MODEL_DIR.resolve()}')
for f in sorted(MODEL_DIR.glob('*.pkl')):
    print(f'    {f.name}')